# T50 - 债市情绪指标计算

## 项目概述

债市情绪指标计算脚本是一个自动化的数据采集和分析工具，用于计算中国债券市场的情绪指标。该指标综合了成交金额和10年国债收益率两个维度的变化，为市场情绪分析提供量化参考。

### 数据源
- **10年国债收益率**: `bond.marketinfo_curve` 表，代码 `L001619604`
- **债券成交金额**: `bond.dealtinfo` 表，按日期汇总

### 核心特性
1. **双维度综合**: 同时考虑成交金额和收益率
2. **方向系数**: 根据收益率方向调整成交金额的贡献
3. **滚动窗口**: 使用近20个交易日的数据计算增长率
4. **两种运行模式**: 支持全量计算和增量更新
5. **UPSERT策略**: 实现幂等性数据更新

---

## 1. 环境配置

### 1.1 导入依赖包

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import logging
from datetime import datetime, timedelta
from typing import Optional, Dict, List, Any

# 添加项目根目录到Python路径（如果需要导入common模块）
project_root = os.path.dirname(os.path.dirname(os.path.dirname(os.path.abspath('__file__'))))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# 尝试导入数据库管理模块
try:
    from common.utils.database import DatabaseManager
    from common.config.database import get_yq_database_config, get_bond_database_config
    HAS_COMMON_MODULE = True
except ImportError:
    HAS_COMMON_MODULE = False
    print("警告: 未找到common模块，将使用独立的数据库连接方式")

print("依赖包导入完成")

### 1.2 配置日志

In [ ]:
# 配置日志格式
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)
logger = logging.getLogger(__name__)

print("日志配置完成")

### 1.3 加载配置参数

In [ ]:
# 从配置文件加载参数
from config import Config

config = Config()

# 显示当前配置
print("=" * 50)
print("当前配置参数:")
print("=" * 50)
print(f"滚动窗口大小: {config.window_size} 天")
print(f"收益率权重: {config.yield_weight}")
print(f"10年国债代码: {config.yield_curve_code}")
print(f"目标表名: {config.target_table}")
print("=" * 50)

### 1.4 建立数据库连接

In [ ]:
def get_database_managers():
    """获取数据库管理器实例"""
    if HAS_COMMON_MODULE:
        # 使用common模块的数据库配置
        yq_db = DatabaseManager(get_yq_database_config())
        bond_db = DatabaseManager(get_bond_database_config())
    else:
        # 使用独立配置
        from sqlalchemy import create_engine
        
        # YQ数据库连接
        yq_config = config.get_yq_db_config()
        yq_url = f"mysql+pymysql://{yq_config['user']}:{yq_config['password']}@{yq_config['host']}:{yq_config['port']}/{yq_config['database']}"
        yq_engine = create_engine(yq_url, pool_recycle=3600)
        
        # Bond数据库连接
        bond_config = config.get_bond_db_config()
        bond_url = f"mysql+pymysql://{bond_config['user']}:{bond_config['password']}@{bond_config['host']}:{bond_config['port']}/{bond_config['database']}"
        bond_engine = create_engine(bond_url, pool_recycle=3600)
        
        # 创建简单的数据库管理器
        class SimpleDBManager:
            def __init__(self, engine):
                self.engine = engine
            
            def execute_query(self, sql, params=None):
                return pd.read_sql(sql, self.engine, params=params)
            
            def execute_sql(self, sql, params=None):
                with self.engine.connect() as conn:
                    conn.execute(sql, params or {})
                    conn.commit()
            
            def table_exists(self, table_name):
                from sqlalchemy import inspect
                inspector = inspect(self.engine)
                return table_name in inspector.get_table_names()
            
            def close(self):
                pass
        
        yq_db = SimpleDBManager(yq_engine)
        bond_db = SimpleDBManager(bond_engine)
    
    return yq_db, bond_db

# 建立数据库连接
yq_db_manager, bond_db_manager = get_database_managers()
logger.info("数据库连接建立成功")

---

## 2. 数据获取

### 2.1 获取10年国债收益率数据

数据来源: `bond.marketinfo_curve` 表
- 代码: `L001619604`（10年国债收益率曲线）
- 字段: `dt`（日期）、`close`（收盘收益率）

In [ ]:
def fetch_yield_data(bond_db, code: str = None) -> pd.DataFrame:
    """
    获取10年国债收益率数据
    
    Args:
        bond_db: 数据库管理器
        code: 收益率曲线代码，默认使用配置中的代码
    
    Returns:
        DataFrame包含dt和yield_10y列
    """
    if code is None:
        code = config.yield_curve_code
    
    logger.info(f"正在获取10年国债收益率数据，代码: {code}")
    
    yield_sql = f"""
    SELECT dt, close as yield_10y 
    FROM bond.marketinfo_curve
    WHERE TRADE_CODE = '{code}'
    ORDER BY dt
    """
    
    yield_df = bond_db.execute_query(yield_sql)
    yield_df['yield_10y'] = pd.to_numeric(yield_df['yield_10y'], errors='coerce')
    yield_df = yield_df.dropna()
    
    logger.info(f"获取到 {len(yield_df)} 条收益率数据")
    
    return yield_df

# 获取收益率数据
yield_df = fetch_yield_data(bond_db_manager)
yield_df.head(10)

In [ ]:
# 查看收益率数据的基本统计信息
print("10年国债收益率数据统计:")
print(yield_df['yield_10y'].describe())
print("\n数据时间范围:")
print(f"起始日期: {yield_df['dt'].min()}")
print(f"结束日期: {yield_df['dt'].max()}")

### 2.2 获取债券成交金额数据

数据来源: `bond.dealtinfo` 表
- 字段: `dt`（日期）、`transaction_amount`（成交金额）
- 聚合: 按日期求和（SUM）

In [ ]:
def fetch_trading_data(bond_db) -> pd.DataFrame:
    """
    获取债券市场成交金额数据
    
    Args:
        bond_db: 数据库管理器
    
    Returns:
        DataFrame包含dt和trading_amount列
    """
    logger.info("正在获取债券市场成交金额数据...")
    
    trading_sql = """
    SELECT dt, SUM(transaction_amount) as trading_amount
    FROM bond.dealtinfo
    GROUP BY dt
    ORDER BY dt
    """
    
    try:
        trading_df = bond_db.execute_query(trading_sql)
        trading_df['trading_amount'] = pd.to_numeric(trading_df['trading_amount'], errors='coerce')
        trading_df = trading_df.dropna()
        logger.info(f"获取到 {len(trading_df)} 条成交金额数据")
    except Exception as e:
        logger.error(f"获取成交金额数据时出错: {e}")
        trading_df = pd.DataFrame(columns=['dt', 'trading_amount'])
    
    return trading_df

# 获取成交金额数据
trading_df = fetch_trading_data(bond_db_manager)
trading_df.head(10)

In [ ]:
# 查看成交金额数据的基本统计信息
print("债券成交金额数据统计:")
print(trading_df['trading_amount'].describe())
print("\n数据时间范围:")
print(f"起始日期: {trading_df['dt'].min()}")
print(f"结束日期: {trading_df['dt'].max()}")

---

## 3. 数据处理

### 3.1 数据合并

使用内连接（inner join）合并两个数据源，确保每个日期都有收益率和成交金额数据。

In [ ]:
def merge_data(yield_df: pd.DataFrame, trading_df: pd.DataFrame) -> pd.DataFrame:
    """
    合并收益率和成交金额数据
    
    Args:
        yield_df: 收益率数据
        trading_df: 成交金额数据
    
    Returns:
        合并后的DataFrame
    """
    logger.info("开始合并数据...")
    
    merged_df = pd.merge(yield_df, trading_df, on='dt', how='inner')
    merged_df = merged_df.sort_values('dt').reset_index(drop=True)
    
    if len(merged_df) == 0:
        logger.warning("没有找到匹配的数据")
    else:
        logger.info(f"合并后得到 {len(merged_df)} 条匹配数据")
    
    return merged_df

# 合并数据
merged_df = merge_data(yield_df, trading_df)
merged_df.head(10)

In [ ]:
# 查看合并后数据的概况
print("合并后数据概况:")
print(f"总记录数: {len(merged_df)}")
print(f"\n数据时间范围:")
print(f"起始日期: {merged_df['dt'].min()}")
print(f"结束日期: {merged_df['dt'].max()}")
print(f"\n数据列信息:")
print(merged_df.dtypes)

---

## 4. 核心逻辑

### 4.1 增长率计算

计算成交金额增长率和收益率增长率：
- 窗口大小: 20个交易日（约1个月）
- 公式: `(当前值 - 起始值) / 起始值`

In [ ]:
def calculate_growth_rate(data: pd.DataFrame, column: str, window_size: int) -> float:
    """
    计算增长率
    
    Args:
        data: 包含历史数据的DataFrame
        column: 要计算增长率的列名
        window_size: 窗口大小
    
    Returns:
        增长率（百分比形式的小数）
    """
    if len(data) < 2:
        return 0.0
    
    first_value = data.iloc[0][column]
    last_value = data.iloc[-1][column]
    
    if first_value == 0:
        return 0.0
    
    return (last_value - first_value) / first_value


# 测试增长率计算
if len(merged_df) >= 20:
    test_data = merged_df.iloc[0:20]
    amount_growth = calculate_growth_rate(test_data, 'trading_amount', 20)
    yield_growth = calculate_growth_rate(test_data, 'yield_10y', 20)
    
    print(f"测试增长率计算（前20条数据）:")
    print(f"成交金额增长率: {amount_growth:.4f} ({amount_growth*100:.2f}%)")
    print(f"收益率增长率: {yield_growth:.4f} ({yield_growth*100:.2f}%)")

### 4.2 情绪指标计算

**公式**:
```
sentiment_index = yield_weight * yield_growth_rate + amount_growth_rate * direction_factor
```

**方向系数 (direction_factor)**:
- 如果 `yield_growth_rate > 0`: `direction_factor = 1`
- 如果 `yield_growth_rate < 0`: `direction_factor = -1`
- 如果 `yield_growth_rate = 0`: `direction_factor = 0`

**设计逻辑**:
- 收益率上升时：成交金额增加是好事（情绪乐观）
- 收益率下降时：成交金额增加是坏事（恐慌抛售）

In [ ]:
def calculate_sentiment_index(
    yield_growth_rate: float,
    amount_growth_rate: float,
    yield_weight: float = 0.5
) -> float:
    """
    计算债市情绪指标
    
    Args:
        yield_growth_rate: 收益率增长率
        amount_growth_rate: 成交金额增长率
        yield_weight: 收益率权重，默认0.5
    
    Returns:
        情绪指标值
    """
    if yield_growth_rate > 0:
        direction_factor = 1
    elif yield_growth_rate < 0:
        direction_factor = -1
    else:
        direction_factor = 0
    
    sentiment_index = yield_weight * yield_growth_rate + amount_growth_rate * direction_factor
    
    return sentiment_index


# 测试情绪指标计算
test_cases = [
    (0.02, 0.1, "收益率上升，成交金额增加"),   # 乐观
    (-0.02, 0.1, "收益率下降，成交金额增加"), # 恐慌
    (0.02, -0.1, "收益率上升，成交金额减少"), # 不确定
    (-0.02, -0.1, "收益率下降，成交金额减少"), # 不确定
]

print("情绪指标计算测试:")
print("=" * 70)
print(f"{'场景':<30} {'收益率增长率':>12} {'成交金额增长率':>15} {'情绪指标':>10}")
print("-" * 70)

for yield_rate, amount_rate, scenario in test_cases:
    sentiment = calculate_sentiment_index(yield_rate, amount_rate)
    print(f"{scenario:<30} {yield_rate:>12.2%} {amount_rate:>15.2%} {sentiment:>10.4f}")

### 4.3 全量计算模式

从历史数据开始计算所有日期的债市情绪指标。

In [ ]:
def calculate_full_sentiment(
    merged_df: pd.DataFrame,
    window_size: int = None,
    yield_weight: float = None
) -> pd.DataFrame:
    """
    全量计算债市情绪指标
    
    Args:
        merged_df: 合并后的数据
        window_size: 滚动窗口大小
        yield_weight: 收益率权重
    
    Returns:
        包含情绪指标结果的DataFrame
    """
    if window_size is None:
        window_size = config.window_size
    if yield_weight is None:
        yield_weight = config.yield_weight
    
    logger.info(f"开始全量计算，窗口大小: {window_size}, 收益率权重: {yield_weight}")
    
    results = []
    
    for i in range(window_size, len(merged_df)):
        # 获取当前日期
        current_date = merged_df.iloc[i]['dt']
        
        # 获取近N个交易日的数据
        recent_data = merged_df.iloc[i-window_size:i].copy()
        
        # 计算成交金额增长率
        amount_growth_rate = calculate_growth_rate(recent_data, 'trading_amount', window_size)
        
        # 计算收益率增长率
        yield_growth_rate = calculate_growth_rate(recent_data, 'yield_10y', window_size)
        
        # 计算情绪指标
        sentiment_index = calculate_sentiment_index(
            yield_growth_rate, amount_growth_rate, yield_weight
        )
        
        # 添加到结果中
        results.append({
            'dt': current_date,
            'trading_amount': merged_df.iloc[i]['trading_amount'],
            'yield_10y': merged_df.iloc[i]['yield_10y'],
            'trading_amount_growth_rate': amount_growth_rate,
            'yield_growth_rate': yield_growth_rate,
            'sentiment_index': sentiment_index
        })
    
    result_df = pd.DataFrame(results)
    logger.info(f"全量计算完成，共 {len(result_df)} 条记录")
    
    return result_df

# 执行全量计算
result_df = calculate_full_sentiment(merged_df)
result_df.head(20)

In [ ]:
# 查看计算结果统计
print("债市情绪指标统计:")
print("=" * 50)
print(result_df['sentiment_index'].describe())
print("\n情绪指标分布:")
print(f"正值（乐观）: {(result_df['sentiment_index'] > 0).sum()} 条")
print(f"负值（悲观）: {(result_df['sentiment_index'] < 0).sum()} 条")
print(f"零值（中性）: {(result_df['sentiment_index'] == 0).sum()} 条")

### 4.4 增量更新模式

只计算昨天的数据，获取近20个交易日的数据用于计算增长率。

In [ ]:
def calculate_yesterday_sentiment(
    bond_db,
    window_size: int = None,
    yield_weight: float = None,
    target_date: str = None
) -> Optional[Dict]:
    """
    增量更新昨日数据
    
    Args:
        bond_db: 数据库管理器
        window_size: 滚动窗口大小
        yield_weight: 收益率权重
        target_date: 目标日期（格式：YYYY-MM-DD），默认为昨天
    
    Returns:
        包含昨日情绪指标的字典，如果数据不足则返回None
    """
    if window_size is None:
        window_size = config.window_size
    if yield_weight is None:
        yield_weight = config.yield_weight
    
    # 计算目标日期
    if target_date is None:
        yesterday = datetime.now() - timedelta(days=1)
        target_date = yesterday.strftime('%Y-%m-%d')
    
    logger.info(f"正在计算 {target_date} 的情绪指标...")
    
    # 获取近N个交易日的收益率数据
    yield_history_sql = f"""
    SELECT dt, close as yield_10y 
    FROM bond.marketinfo_curve
    WHERE TRADE_CODE = '{config.yield_curve_code}' AND dt <= '{target_date}'
    ORDER BY dt DESC
    LIMIT {window_size}
    """
    yield_history = bond_db.execute_query(yield_history_sql)
    yield_history['yield_10y'] = pd.to_numeric(yield_history['yield_10y'], errors='coerce')
    yield_history = yield_history.dropna().sort_values('dt').reset_index(drop=True)
    
    # 获取近N个交易日的成交金额数据
    trading_history_sql = f"""
    SELECT dt, SUM(transaction_amount) as trading_amount
    FROM bond.dealtinfo
    WHERE dt <= '{target_date}'
    GROUP BY dt
    ORDER BY dt DESC
    LIMIT {window_size}
    """
    trading_history = bond_db.execute_query(trading_history_sql)
    trading_history['trading_amount'] = pd.to_numeric(trading_history['trading_amount'], errors='coerce')
    trading_history = trading_history.dropna().sort_values('dt').reset_index(drop=True)
    
    # 合并数据
    merged_history = pd.merge(yield_history, trading_history, on='dt', how='inner')
    merged_history = merged_history.sort_values('dt').reset_index(drop=True)
    
    if len(merged_history) < 2:
        logger.warning(f"历史数据不足，至少需要2条数据，当前有 {len(merged_history)} 条")
        return None
    
    # 获取目标日期的数据
    target_data = merged_history[merged_history['dt'] == target_date]
    if len(target_data) == 0:
        # 使用最新的数据
        target_data = merged_history.iloc[[-1]]
        logger.info(f"未找到 {target_date} 的数据，使用最新日期 {target_data.iloc[0]['dt']} 的数据")
    
    yield_10y = target_data.iloc[0]['yield_10y']
    trading_amount = target_data.iloc[0]['trading_amount']
    
    # 计算增长率
    amount_growth_rate = calculate_growth_rate(merged_history, 'trading_amount', window_size)
    yield_growth_rate = calculate_growth_rate(merged_history, 'yield_10y', window_size)
    
    logger.info(f"成交金额增长率: {amount_growth_rate:.6f}")
    logger.info(f"收益率增长率: {yield_growth_rate:.6f}")
    
    # 计算情绪指标
    sentiment_index = calculate_sentiment_index(
        yield_growth_rate, amount_growth_rate, yield_weight
    )
    
    logger.info(f"情绪指标: {sentiment_index:.6f}")
    
    return {
        'dt': target_data.iloc[0]['dt'],
        'trading_amount': float(trading_amount),
        'yield_10y': float(yield_10y),
        'trading_amount_growth_rate': float(amount_growth_rate),
        'yield_growth_rate': float(yield_growth_rate),
        'sentiment_index': float(sentiment_index)
    }

# 测试增量更新（使用最近的日期）
yesterday_result = calculate_yesterday_sentiment(bond_db_manager)
if yesterday_result:
    print("\n昨日情绪指标计算结果:")
    for key, value in yesterday_result.items():
        print(f"  {key}: {value}")

---

## 5. 执行与测试

### 5.1 创建目标表

如果表不存在，创建 `bond_market_sentiment` 表。

In [ ]:
def ensure_table_exists(yq_db, table_name: str = None):
    """
    确保目标表存在
    
    Args:
        yq_db: 数据库管理器
        table_name: 表名
    """
    if table_name is None:
        table_name = config.target_table
    
    if not yq_db.table_exists(table_name):
        create_table_query = f"""
        CREATE TABLE {table_name} (
            dt DATE PRIMARY KEY,
            trading_amount DECIMAL(20, 6),
            yield_10y DECIMAL(10, 6),
            trading_amount_growth_rate DECIMAL(10, 6),
            yield_growth_rate DECIMAL(10, 6),
            sentiment_index DECIMAL(10, 6)
        )
        """
        yq_db.execute_sql(create_table_query)
        logger.info(f"已创建 {table_name} 表")
    else:
        logger.info(f"表 {table_name} 已存在")

# 确保表存在
ensure_table_exists(yq_db_manager)

### 5.2 数据存储（UPSERT策略）

In [ ]:
def save_sentiment_data(yq_db, data: pd.DataFrame, table_name: str = None):
    """
    保存情绪指标数据到数据库（使用UPSERT策略）
    
    Args:
        yq_db: 数据库管理器
        data: 要保存的数据（DataFrame或字典）
        table_name: 表名
    """
    if table_name is None:
        table_name = config.target_table
    
    insert_sql = f"""
    INSERT INTO {table_name} 
    (dt, trading_amount, yield_10y, trading_amount_growth_rate, yield_growth_rate, sentiment_index)
    VALUES (:dt, :trading_amount, :yield_10y, :trading_amount_growth_rate, :yield_growth_rate, :sentiment_index)
    ON DUPLICATE KEY UPDATE
    trading_amount = VALUES(trading_amount),
    yield_10y = VALUES(yield_10y),
    trading_amount_growth_rate = VALUES(trading_amount_growth_rate),
    yield_growth_rate = VALUES(yield_growth_rate),
    sentiment_index = VALUES(sentiment_index)
    """
    
    if isinstance(data, pd.DataFrame):
        # 批量插入
        logger.info(f"开始插入 {len(data)} 条记录到数据库...")
        for _, row in data.iterrows():
            yq_db.execute_sql(insert_sql, row.to_dict())
        logger.info(f"成功更新 {len(data)} 条记录")
    elif isinstance(data, dict):
        # 单条插入
        logger.info(f"开始插入单条记录到数据库...")
        yq_db.execute_sql(insert_sql, data)
        logger.info(f"成功更新 1 条记录")
    else:
        raise ValueError("数据类型必须是DataFrame或dict")

# 保存全量计算结果
if len(result_df) > 0:
    save_sentiment_data(yq_db_manager, result_df)

### 5.3 验证结果

In [ ]:
def verify_results(yq_db, table_name: str = None, limit: int = 10):
    """
    验证数据库中的结果
    
    Args:
        yq_db: 数据库管理器
        table_name: 表名
        limit: 返回的记录数
    """
    if table_name is None:
        table_name = config.target_table
    
    verify_sql = f"""
    SELECT * FROM {table_name}
    ORDER BY dt DESC
    LIMIT {limit}
    """
    
    result = yq_db.execute_query(verify_sql)
    
    logger.info(f"验证结果，最近 {limit} 条记录:")
    
    return result

# 验证结果
verify_df = verify_results(yq_db_manager)
verify_df

In [ ]:
# 查询数据库中的记录总数
count_sql = f"SELECT COUNT(*) as total FROM {config.target_table}"
count_result = yq_db_manager.execute_query(count_sql)
print(f"数据库中总记录数: {count_result.iloc[0]['total']}")

### 5.4 关闭数据库连接

In [ ]:
# 关闭数据库连接
yq_db_manager.close()
bond_db_manager.close()
logger.info("数据库连接已关闭")
print("\n所有操作完成！")

---

## 6. 工具函数

以下是可以复用的辅助函数。

In [ ]:
def run_full_calculation():
    """
    执行全量计算的主函数
    
    可以在其他脚本中调用此函数来执行全量计算
    """
    logger.info("=" * 50)
    logger.info("开始执行全量计算")
    logger.info("=" * 50)
    
    try:
        # 获取数据库连接
        yq_db, bond_db = get_database_managers()
        
        # 获取数据
        yield_df = fetch_yield_data(bond_db)
        trading_df = fetch_trading_data(bond_db)
        
        # 合并数据
        merged_df = merge_data(yield_df, trading_df)
        
        if len(merged_df) == 0:
            logger.warning("没有数据可处理")
            return
        
        # 计算情绪指标
        result_df = calculate_full_sentiment(merged_df)
        
        # 保存结果
        ensure_table_exists(yq_db)
        save_sentiment_data(yq_db, result_df)
        
        logger.info("全量计算完成")
        
    except Exception as e:
        logger.error(f"全量计算出错: {e}")
        raise
    finally:
        yq_db.close()
        bond_db.close()


def run_incremental_update(target_date: str = None):
    """
    执行增量更新的主函数
    
    Args:
        target_date: 目标日期（格式：YYYY-MM-DD），默认为昨天
    
    可以在其他脚本中调用此函数来执行增量更新
    """
    logger.info("=" * 50)
    logger.info("开始执行增量更新")
    logger.info("=" * 50)
    
    try:
        # 获取数据库连接
        yq_db, bond_db = get_database_managers()
        
        # 计算昨日情绪指标
        result = calculate_yesterday_sentiment(bond_db, target_date=target_date)
        
        if result is None:
            logger.warning("无法计算情绪指标")
            return
        
        # 保存结果
        ensure_table_exists(yq_db)
        save_sentiment_data(yq_db, result)
        
        logger.info("增量更新完成")
        
    except Exception as e:
        logger.error(f"增量更新出错: {e}")
        raise
    finally:
        yq_db.close()
        bond_db.close()


print("工具函数定义完成")

In [ ]:
# 使用示例

# 全量计算
# run_full_calculation()

# 增量更新（昨天）
# run_incremental_update()

# 增量更新（指定日期）
# run_incremental_update('2024-01-15')

print("取消注释上述代码以执行相应操作")

---

## 附录：命令行运行方式

如果需要通过命令行运行，可以使用以下方式：

```bash
# 全量计算模式
python 债市情绪指标计算.py --mode full

# 增量更新模式（更新昨日数据）
python 债市情绪指标计算.py --mode yesterday
```